# 灰色 GM(2, 1) 预测模型

例 15.4 已知 $ \boldsymbol{x}^{(0)} = (41, 49, 61, 78, 96, 104) $，试建立 $ \text{GM}(2,1) $ 模型.  

解 $ \boldsymbol{x}^{(0)} $ 的 1-AGO 序列 $ \boldsymbol{x}^{(1)} $ 和 1-IAGO 序列 $ \alpha^{(1)}\boldsymbol{x}^{(0)} $ 分别为  
$$ 
\boldsymbol{x}^{(1)} = (41, 90, 151, 229, 325, 429), \quad \alpha^{(1)}\boldsymbol{x}^{(0)} = (8, 12, 17, 18, 8). 
$$  

$ \boldsymbol{x}^{(1)} $ 的均值生成序列 $ \boldsymbol{z}^{(1)} = (65.5, 120.5, 190, 277, 377) $，记  
$$
\boldsymbol{B} = \begin{bmatrix} 
-x^{(0)}(2) & -z^{(1)}(2) & 1 \\
-x^{(0)}(3) & -z^{(1)}(3) & 1 \\
\vdots & \vdots & \vdots \\
-x^{(0)}(6) & -z^{(1)}(6) & 1 
\end{bmatrix} = \begin{bmatrix} 
-49 & -65.5 & 1 \\
-61 & -120.5 & 1 \\
-78 & -190 & 1 \\
-96 & -277 & 1 \\
-104 & -377 & 1 
\end{bmatrix}, \quad 
\boldsymbol{Y} = \begin{bmatrix} 8 \\ 12 \\ 17 \\ 18 \\ 8 \end{bmatrix},
$$  

则有  
$$
\hat{\boldsymbol{u}} = \begin{bmatrix} \hat{a}_1 \\ \hat{a}_2 \\ \hat{b} \end{bmatrix} = (\boldsymbol{B}^\text{T}\boldsymbol{B})^{-1}\boldsymbol{B}^\text{T}\boldsymbol{Y} = \begin{bmatrix} -1.0922 \\ 0.1959 \\ -31.7983 \end{bmatrix},
$$  

故得 $ \text{GM}(2,1) $ 白化模型  
$$
\frac{d^2 x^{(1)}}{dt^2} - 1.0922 \frac{dx^{(1)}}{dt} + 0.1959 x^{(1)} = -31.7983.
$$  

利用边值条件 $ x^{(1)}(1) = 41 $, $ x^{(1)}(6) = 429 $, 解之得时间响应式为  
$$
x^{(1)}(t) = 203.85e^{0.22622t} - 0.5325e^{0.86597t} - 162.317,
$$  

于是 $ \text{GM}(2,1) $ 时间响应式  
$$
\hat{x}^{(1)}(k + 1) = 203.85e^{0.22622k} - 0.5325e^{0.86597k} - 162.317.
$$  

所以  
$$
\hat{\boldsymbol{x}}^{(1)} = (41, 92.0148, 155.1561, 232.3672, 324.5220, 429).
$$  

做 IAGO 还原, 有  
$$
\hat{\boldsymbol{x}}^{(0)} = (41, 51.0148, 63.1412, 77.2111, 92.1548, 104.4780).
$$  

计算结果见表 15.4.  


**表 15.4 误差检验表**  

| 序号 | 实际数据 $ x^{(0)} $ | 预测数据 $ \hat{x}^{(0)} $ | 残差 $ x^{(0)} - \hat{x}^{(0)} $ | 相对误差 $ \delta(k)/\% $ |
| :---: | :---: | :---: | :---: | :---: |
| 2 | 49 | 51.0148 | -2.0148 | 4.1 |
| 3 | 61 | 63.1412 | -2.1412 | 3.5 |
| 4 | 78 | 77.2111 | 0.7889 | 1.0 |
| 5 | 96 | 92.1548 | 3.8452 | 4.0 |
| 6 | 104 | 104.4780 | -0.4780 | 0.5 |

In [ ]:
# 通过特征方程和待定系数法求解二阶微分方程
import numpy as np
import sympy as sp

x0 = np.array([41, 49, 61, 78, 96, 104])
n = len(x0)
x1 = np.cumsum(x0)  # 1-AGO 1次累加操作
ax0 = np.diff(x0)  # 1-IAGO 1次逆累加操作
z = 0.5 * (x1[1:] + x1[:-1])  # 均值生成数列
B = np.c_[-x0[1:], -z, np.ones((n-1, 1))]
Y = ax0
u = np.linalg.pinv(B) @ Y  # 系数，B^+ @ Y

p = np.r_[1, u[:-1]]  # 构造特征方程的特征多项式(系数)，(1, a1, a2)，二次
r = np.roots(p)  # 求特征根，两个
special_solution = u[2] / u[1]  # 二阶常系数微分方程的特解, b / a2即 u[-1] / u[-2]或 u[2] / u[1]
c1, c2, t = sp.symbols('c1 c2 t')
# 方程组求非齐次通解c1 * e^(r1*x) + c2 * e(r2*x)的未知数c1, c2
eq1 = c1 + c2 + special_solution -41  # 边值条件 x = 0
eq2 = c1 * np.exp(5 * r[0]) + c2 * np.exp(5 * r[1]) + special_solution - 429  # 边值条件 x = 5
c = sp.solve([eq1, eq2], [c1, c2])  # 解方程组得到c1, c2
s = c[c1] * sp.exp(r[0] * t) + c[c2] * sp.exp(r[1] * t) + special_solution  # 微分方程的符号解
x_AGO_pred = []  # 存累加数列预测值, (6,)
for i in range(6): 
    x_AGO_pred.append(s.subs({t: i}))
x0_pred = np.r_[x_AGO_pred[0], np.diff(x_AGO_pred)]  # (6,)
difference = x0 - x0_pred  # 残差, (6,)
delta = np.abs(difference) / x0  # 相对误差, (6,)
print(f"预测值(累加数列):\n{x_AGO_pred}")
print(f"预测值(原始数据):\n{x0_pred}")
print(f"残差:\n{difference}")
print(f"相对误差:\n{delta}")
print(f"二阶常系数非齐次线性微分方程的通解为:\n{s}")

预测值(累加数列):
[40.9999999999997, 92.0148144078681, 155.156052197627, 232.367192369488, 324.521982077495, 429.000000000000]
预测值(原始数据):
[40.9999999999997 51.0148144078684 63.1412377897588 77.2111401718613
 92.1547897080068 104.478017922505]
残差:
[2.84217094304040e-13 -2.01481440786836 -2.14123778975878
 0.788859828138726 3.84521029199323 -0.478017922504989]
相对误差:
[6.93212425131805e-15 0.0411186613850685 0.0351022588485045
 0.0101135875402401 0.0400542738749294 0.00459632617793259]
二阶常系数非齐次线性微分方程的通解为:
203.849012866397*exp(0.22622340416904*t) - 0.532505769427844*exp(0.865972945416792*t) - 162.316507096969


In [33]:
# 直接求二阶微分方程符号解
import numpy as np
import sympy as sp

x0 = np.array([41, 49, 61, 78, 96, 104])
n = len(x0)
ratio = x0[:-1] / x0[1:]  # 计算级比
uplow_bound = [ratio.min(), ratio.max()]
bound_permit = [np.exp(-2 / (n + 1)), np.exp(2 / (n + 1))]  # 级比容许范围
x1 = np.cumsum(x0)
z = (x1[1:] + x1[:-1]) * 0.5
B = np.vstack([-x0[1:], -z, np.ones(n-1)]).T
u = np.linalg.pinv(B) @ np.diff(x0)  # 最小二乘法拟合得到参数
print(f"(a1, a2, b) = {u}")

sp.var('t')
sp.var('x', cls=sp.Function)
eq = x(t).diff(t, 2) + u[0] * x(t).diff(t) + u[1] * x(t) - u[2]
s = sp.dsolve(eq, ics={x(0): x1[0], x(5): x1[-1]})  # 求解微分方程的符号解，注意这是求累加序列,故使用x1
print(f"{s.args[0]}={s.args[1]}")
fxt = sp.lambdify(t, s.args[1], 'numpy')  # 转换为匿名函数
x_AGO_pred = fxt(np.arange(n))  # 累加序列的预测值
x_pred = np.hstack([x0[0], np.diff(x_AGO_pred)])  # 参考序列的预测值，不能用c_
difference = x0 - x_pred  # 残差
delta = np.abs(difference) / x0  # 相对误差
print(f"残差:\n{difference}")
print(f"相对误差:\n{np.round(delta*100, 2)}")

(a1, a2, b) = [ -1.09219635   0.19590335 -31.79834712]
x(t)=203.849012866397*exp(0.22622340416904*t) - 0.532505769427845*exp(0.865972945416789*t) - 162.31650709697
残差:
[ 0.         -2.01481441 -2.14123779  0.78885983  3.84521029 -0.47801792]
相对误差:
[0.   4.11 3.51 1.01 4.01 0.46]
